# Modélisation : filtrage collaboratif (ALS)

Même protocole que le notebook 02.

In [1]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"

import json
import sys
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight
from surprise import SVD, Dataset, Reader

sys.path.insert(0, "../src")
from utils import N_RECO, SEED, hit_rate_at_k, load_articles_metadata, load_clicks, mrr_at_k, sample_eval_users, temporal_split

## Chargement et split

In [2]:
clicks = load_clicks("../data")
train, test = temporal_split(clicks)
users_eval = sample_eval_users(test, n=5000, seed=SEED)

test_items = test[test["user_id"].isin(users_eval)].set_index("user_id")["click_article_id"].to_dict()

print("Clics train :", len(train))
print("Clics test :", len(test))
print("Utilisateurs évalués :", len(users_eval))

Clics train : 2665284
Clics test : 322897
Utilisateurs évalués : 5000


## 1. Matrice utilisateurs x articles

In [3]:
articles = load_articles_metadata("../data")
n_articles = len(articles)
assert articles["article_id"].min() == 0 and articles["article_id"].max() == n_articles - 1

user_ids = np.sort(train["user_id"].unique())
lignes = np.searchsorted(user_ids, train["user_id"].to_numpy())
colonnes = train["click_article_id"].to_numpy()
assert (user_ids[lignes] == train["user_id"].to_numpy()).all()

mat_train = sp.csr_matrix((np.ones(len(train), dtype=np.float32), (lignes, colonnes)), shape=(len(user_ids), n_articles))
assert int(mat_train.sum()) == len(train)

for u in np.random.default_rng(SEED).choice(users_eval, size=5, replace=False):
    i = int(np.searchsorted(user_ids, u))
    clics_u = train.loc[train["user_id"] == u, "click_article_id"]
    assert user_ids[i] == u
    assert set(mat_train[i].indices) == set(clics_u)
    assert int(mat_train[i].data.sum()) == len(clics_u)

indices_eval = np.searchsorted(user_ids, users_eval)
assert (user_ids[indices_eval] == users_eval).all()

articles_cliques = train["click_article_id"].nunique()
print("Matrice train :", mat_train.shape[0], "utilisateurs x", mat_train.shape[1], "articles")
print("Cases non nulles :", mat_train.nnz)
print("Articles avec au moins un clic dans le train :", articles_cliques, "| sans clic :", n_articles - articles_cliques)

Matrice train : 322897 utilisateurs x 364047 articles
Cases non nulles : 2629418
Articles avec au moins un clic dans le train : 41002 | sans clic : 323045


## 2. ALS (implicit)

La bibliothèque surprise est faite pour des notes explicites : on lui donne 1 pour un clic et 0 pour un article non lu. La SVD est testée en section 5.

In [4]:
als = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=15, random_state=SEED)
t0 = time.time()
als.fit(mat_train, show_progress=False)
print("Entraînement ALS :", round(time.time() - t0, 1), "s")

Entraînement ALS : 20.2 s


## 3. Top 5 et métriques

In [5]:
def evaluer(recos):
    m = {"hit_rate_at_5": hit_rate_at_k(recos, test_items), "mrr_at_5": mrr_at_k(recos, test_items)}
    print("Hit rate :", round(m["hit_rate_at_5"], 4), "| MRR @ 5 :", round(m["mrr_at_5"], 4))
    return m


def recommander_als(model, mat):
    recos = {}
    for debut in range(0, len(users_eval), 500):
        bloc = indices_eval[debut:debut + 500]
        ids, _ = model.recommend(bloc, mat[bloc], N=N_RECO, filter_already_liked_items=True)
        for j, u in enumerate(users_eval[debut:debut + 500]):
            recos[u] = [int(a) for a in ids[j]]
    return recos


metriques_als = evaluer(recommander_als(als, mat_train))

Hit rate : 0.1388 | MRR @ 5 : 0.0779


## 4. Variante : pondération BM25

bm25_weight baisse le poids des articles que tout le monde clique et normalise par l'activité de chaque utilisateur, puis on réentraîne le même ALS sur la matrice pondérée.

In [6]:
mat_bm25 = bm25_weight(mat_train, K1=100, B=0.8).tocsr().astype(np.float32)

als_bm25 = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=15, random_state=SEED)
t0 = time.time()
als_bm25.fit(mat_bm25, show_progress=False)
print("Entraînement ALS + BM25 :", round(time.time() - t0, 1), "s")

metriques_bm25 = evaluer(recommander_als(als_bm25, mat_bm25))

Entraînement ALS + BM25 : 22.2 s


Hit rate : 0.2166 | MRR @ 5 : 0.1162


## 5. Variante : SVD (surprise)

surprise attend des notes explicites : on lui donne 1 pour chaque clic et 0 pour un article non lu tiré au hasard parmi les articles du train (autant de 0 que de 1), sinon toutes les notes valent 1 et le modèle n'a rien à classer.

Pour le top 5, algo.predict est trop lent (5 000 x 41 000 couples) : on refait son calcul (moyenne + biais + produit des facteurs) en numpy par paquets de 500 utilisateurs, articles déjà lus exclus.

In [7]:
coo = mat_train.tocoo()
articles_train = np.unique(coo.col)
rng = np.random.default_rng(SEED)
negatifs = rng.choice(articles_train, size=len(coo.row))
deja_lu = np.asarray(mat_train[coo.row, negatifs]).ravel() > 0
while deja_lu.any():
    negatifs[deja_lu] = rng.choice(articles_train, size=int(deja_lu.sum()))
    deja_lu = np.asarray(mat_train[coo.row, negatifs]).ravel() > 0

notes = pd.DataFrame({
    "user_id": np.tile(user_ids[coo.row], 2),
    "article_id": np.concatenate([coo.col, negatifs]),
    "note": np.repeat([1.0, 0.0], len(coo.row)),
})
trainset = Dataset.load_from_df(notes, Reader(rating_scale=(0, 1))).build_full_trainset()
print("Notes :", trainset.n_ratings, "| utilisateurs :", trainset.n_users, "| articles :", trainset.n_items)

svd = SVD(n_factors=64, n_epochs=20, random_state=SEED)
t0 = time.time()
svd.fit(trainset)
print("Entraînement SVD (surprise) :", round(time.time() - t0, 1), "s")

inner_users = np.array([trainset.to_inner_uid(int(u)) for u in users_eval])
raw_articles = np.array([trainset.to_raw_iid(i) for i in range(trainset.n_items)])
recos_svd = {}
for debut in range(0, len(users_eval), 500):
    bloc = inner_users[debut:debut + 500]
    scores = trainset.global_mean + svd.bu[bloc][:, None] + svd.bi[None, :] + svd.pu[bloc] @ svd.qi.T
    for j, u in enumerate(users_eval[debut:debut + 500]):
        lus = [trainset.to_inner_iid(int(a)) for a in mat_train[indices_eval[debut + j]].indices]
        scores[j, lus] = -np.inf
        top = np.argpartition(-scores[j], N_RECO)[:N_RECO]
        recos_svd[u] = [int(raw_articles[a]) for a in top[np.argsort(-scores[j, top])]]

metriques_svd = evaluer(recos_svd)


Notes : 5258836 | utilisateurs : 322897 | articles : 41002


Entraînement SVD (surprise) : 22.2 s


Hit rate : 0.003 | MRR @ 5 : 0.0011


## Comparaison

Les métriques du notebook 02 sont rechargées (même protocole). La SVD (surprise) fait 0,0030 de hit rate, sous la popularité (0,0348) et loin de l'ALS + BM25 (0,2166).

In [8]:
with open("../results/metrics_02.json", encoding="utf-8") as f:
    metriques_02 = json.load(f)

noms = {
    "popularite": "Popularité",
    "content_based_250": "Content-based 250 d",
    "content_based_dernier": "Content-based dernier article lu",
    "content_based_pca50": "Content-based PCA 50 d",
}
comparaison = {noms.get(k, k): v for k, v in metriques_02.items()}
comparaison.update({"ALS": metriques_als, "ALS + BM25": metriques_bm25, "SVD (surprise)": metriques_svd})

metriques_03 = {"als": metriques_als, "als_bm25": metriques_bm25, "svd": metriques_svd}
with open("../results/metrics_03.json", "w", encoding="utf-8") as f:
    json.dump(metriques_03, f, indent=2)

pd.DataFrame(comparaison).T.round(4)

,hit_rate_at_5,mrr_at_5
Popularité,0.0348,0.0157
Content-based 250 d,0.0088,0.0048
Content-based dernier article lu,0.0130,0.0070
Content-based PCA 50 d,0.0068,0.0040
ALS,0.1388,0.0779
ALS + BM25,0.2166,0.1162
SVD (surprise),0.0030,0.0011


## Conclusion

- Le collaboratif fait nettement mieux que les autres modèles : hit rate @ 5 de 0,2166 pour l'ALS + BM25, soit 6 fois la popularité (0,0348) et plus de 30 fois le content-based PCA 50 d (0,0068).
- La pondération BM25 aide beaucoup (de 0,1388 à 0,2166) : elle baisse le poids des articles que tout le monde clique.
- La SVD (surprise) sur les mêmes clics fait 0,0030, sous la popularité : ses 0 sont des articles non lus tirés au hasard et traités comme de vrais rejets, alors que l'ALS pondère chaque case par une confiance. Avec 2 ou 3 notes par utilisateur, les facteurs restent proches de leur tirage initial et leur produit noie les biais.
- Limite : l'ALS ne peut recommander que les 41 002 articles vus dans le train (sur 364 047), et un article neuf demande un réentraînement complet.
- Le content-based couvre tout le catalogue (l'embedding d'un article neuf suffit) : c'est le content-based PCA 50 d, léger (73 Mo), qui avait été déployé en v1.
- Attention, ce protocole a une fuite temporelle : en médiane 25 % des clics du train sont postérieurs au clic à prédire, et la cible est le clic suivant de la même session, ce qui gonfle les scores du collaboratif.
- Le notebook 04 refait l'évaluation sans fuite : l'ALS + BM25 s'y effondre et le modèle retenu est l'hybride tendance + co-visitation + content-based.